In [1]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# ==========================================
# 1. SETUP AMBIENTE E VERIFICA DIRECTORY
# ==========================================
TESTO_DIR = "key_results_testi"
OUTPUT_CSV_ANONIMO = "dataset_report_anonimizzati_llama_cpp.csv"

if not os.path.exists(TESTO_DIR):
    raise FileNotFoundError(f"La cartella '{TESTO_DIR}' non esiste nella directory corrente.")

# ==========================================
# 2. ESTRAZIONE METADATI E PULIZIA TESTO
# ==========================================
def estrai_metadata(nome_file):
    base = nome_file.replace("_KeyResults.txt", "")
    periodo_match = re.search(r'([A-Za-z]{3}_\d{4})_-_([A-Za-z]{3}_\d{4})', base)

    if periodo_match:
        inizio_periodo = periodo_match.group(1).replace('_', ' ')
        fine_periodo = periodo_match.group(2).replace('_', ' ')
        periodo = f"{inizio_periodo} / {fine_periodo}"
    else:
        inizio_periodo = None
        fine_periodo = None
        periodo = base

    paese_match = re.match(r'^(.+?)_[A-Za-z]{3}_\d{4}', base)
    paese = paese_match.group(1).replace('_', ' ') if paese_match else base

    return {
        "paese": paese,
        "periodo": periodo,
        "inizio_periodo": inizio_periodo,
        "fine_periodo": fine_periodo,
        "nome_file": nome_file
    }

# ==========================================
# 3. DOWNLOAD GGUF E INIZIALIZZAZIONE LLAMA.CPP
# ==========================================
print("Recupero dei pesi quantizzati GGUF (Llama-3.2-3B-Instruct, 4-bit)...")
modello_gguf_path = hf_hub_download(
    repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF",
    filename="Llama-3.2-3B-Instruct-Q4_K_M.gguf"
)

print("Inizializzazione del motore di inferenza MPS (Metal)...")
llm = Llama(
    model_path=modello_gguf_path,
    n_gpu_layers=-1,  # Offload totale: scarica tutti i livelli sulla GPU Apple
    n_ctx=4096,       # Finestra di contesto rigida
    verbose=False     # Soppressione dei log diagnostici C++ a schermo
)

# ==========================================
# 4. GESTIONE DEI PROMPT SEMANTICI
# ==========================================
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain paragraphs. Each paragraph MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "CRITICAL RULE 5: Output Language. You MUST write the entire output in English, even if the input report is written in French, Spanish, or any other language.\n"
    "The 4 required tags are:\n"
    "- [SHOCKS AND DRIVERS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged sections. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Section 1 must start with [SHOCKS AND DRIVERS]: and focus on ALL underlying causes: economic factors, agriculture, climate shocks (rainfall, droughts), conflict, displacement, and disease outbreaks.\n"
    "2) Section 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Section 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Section 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods, nutrition, and displacement consequences.\n\n"
    "Report to analyze:\n"
)

# ==========================================
# 5. ELABORAZIONE DEL BATCH DI FILE
# ==========================================
files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"\nTrovati {len(files)} report in formato .txt da elaborare...\n")

for nome_file in tqdm(files, desc="Elaborazione report"):

    if os.path.exists(OUTPUT_CSV_ANONIMO):
        df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
        if nome_file in df_check['nome_file'].values:
            continue

    path_file = os.path.join(TESTO_DIR, nome_file)
    with open(path_file, "r", encoding="utf-8") as f:
        testo = f.read()

    testo_pulito = re.sub(r'^.*?={20,}\n*', '', testo, flags=re.DOTALL).strip()

    if not testo_pulito:
        print(f" -> Avviso: il file {nome_file} risulta vuoto dopo la pulizia.")
        continue

    metadata = estrai_metadata(nome_file)

    # Iniezione del prompt nel formato chat nativo
    messages = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
    ]

    try:
        # Inferenza quantizzata C++
        outputs = llm.create_chat_completion(
            messages=messages,
            max_tokens=700,
            temperature=0.1
        )

        scheda_anonima = outputs["choices"][0]["message"]["content"].strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        nuovo_dato = pd.DataFrame([{
            "nome_file": nome_file,
            "paese": metadata["paese"],
            "periodo": metadata["periodo"],
            "inizio_periodo": metadata["inizio_periodo"],
            "fine_periodo": metadata["fine_periodo"],
            "testo_originale": testo_pulito,
            "report_anonimo": scheda_anonima
        }])

        nuovo_dato.to_csv(OUTPUT_CSV_ANONIMO, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV_ANONIMO), encoding='utf-8')

    except Exception as e:
        print(f"\nErrore di esecuzione sul file {nome_file}: {e}")
        time.sleep(2)

print(f"\nPipeline completata. Output salvato in: '{OUTPUT_CSV_ANONIMO}'")

/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Recupero dei pesi quantizzati GGUF (Llama-3.2-3B-Instruct, 4-bit)...


Inizializzazione del motore di inferenza MPS (Metal)...

Trovati 497 report in formato .txt da elaborare...



Elaborazione report: 100%|██████████| 497/497 [46:15<00:00,  5.58s/it]  


Pipeline completata. Output salvato in: 'dataset_report_anonimizzati_llama_cpp.csv'
